# LGBM Regression Training - Spearman First

Train deployable LightGBM regression models for the long, short, and signed quality targets.
The notebook uses purge-aware walk-forward validation, tunes by Spearman, and exports a model pack
that can be consumed by the production stack.

In [6]:
import inspect
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import scipy.stats as stats
import lightgbm as lgb
from IPython.display import display
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from Learn.features import add_feature_library
from Learn.labels import calculate_trade_outcomes_all_candles
from Learn.preprocess import preprocess_ohlcv

warnings.filterwarnings("ignore")


def load_ohlcv(ds_name, n_rows=None):
    """Load and prepare OHLCV data from a CSV file."""
    df = pd.read_csv(ds_name)
    if n_rows is not None:
        df = df.tail(n_rows)
    df = df.sort_values("Time").reset_index(drop=True)
    df["Time"] = pd.to_datetime(df["Time"])
    return df


def safe_spearman(y_true, y_pred):
    """Compute Spearman rank correlation with safe fallback to 0.0."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2 or np.nanstd(y_true) == 0.0 or np.nanstd(y_pred) == 0.0:
        return 0.0
    value = stats.spearmanr(y_true, y_pred, nan_policy="omit").statistic
    return 0.0 if value is None or np.isnan(value) else float(value)


def reg_metrics(y_true, y_pred):
    """Compute a dict of standard regression metrics."""
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": r2_score(y_true, y_pred),
        "Spearman": safe_spearman(y_true, y_pred),
    }


def lgbm_spearman_eval(y_true, y_pred):
    """Custom LightGBM eval function tracking Spearman (higher is better)."""
    return "spearman", safe_spearman(y_true, y_pred), True


def walk_forward_folds(n_samples, min_train_size, test_size, gap_size, n_folds):
    """
    Generate expanding-window walk-forward train/val indices.

    Each fold uses all data up to train_end for training, then
    a contiguous test block separated by a gap.
    """
    folds = []
    train_end = int(min_train_size)
    for _ in range(int(n_folds)):
        test_start = train_end + int(gap_size)
        if test_start >= n_samples:
            break
        test_end = min(test_start + int(test_size), n_samples)
        if test_end <= test_start:
            break
        train_idx = np.arange(0, train_end, dtype=np.int64)
        test_idx = np.arange(test_start, test_end, dtype=np.int64)
        if len(train_idx) == 0 or len(test_idx) == 0:
            break
        folds.append((train_idx, test_idx))
        train_end = test_end
        if train_end >= n_samples:
            break
    return folds


def describe_target_frame(frame, target_cols):
    """Return a summary DataFrame of target distributions."""
    rows = []
    for col in target_cols:
        s = pd.Series(frame[col])
        rows.append({
            "target": col,
            "mean": float(s.mean()),
            "std": float(s.std()),
            "p01": float(s.quantile(0.01)),
            "p50": float(s.quantile(0.50)),
            "p99": float(s.quantile(0.99)),
        })
    return pd.DataFrame(rows).set_index("target")


def tune_lgbm_by_spearman(
    X,
    y,
    folds,
    candidate_params,
    base_params,
    sample_weight=None,
    early_stopping_rounds=200,
    robustness_penalty=0.25,
):
    """
    Tune hyper-parameter candidates via walk-forward cross-validation.

    Ranks candidates by a robust score = mean Spearman - penalty * std Spearman,
    then mean Spearman, then mean R². Returns (results_df, best_cfg_dict).
    """
    rows = []
    for cfg_i, cfg in enumerate(candidate_params, start=1):
        fold_rows = []
        best_iters = []
        for tr_idx, va_idx in folds:
            model = LGBMRegressor(**{**base_params, **cfg})
            fit_kwargs = {
                "X": X[tr_idx],
                "y": y[tr_idx],
                "eval_set": [(X[va_idx], y[va_idx])],
                "eval_metric": lgbm_spearman_eval,
                "callbacks": [lgb.early_stopping(early_stopping_rounds, verbose=False)],
            }
            if sample_weight is not None:
                fit_kwargs["sample_weight"] = sample_weight[tr_idx]
            model.fit(**fit_kwargs)

            best_iter = getattr(model, "best_iteration_", None)
            if best_iter is None or best_iter <= 0:
                best_iter = model.n_estimators
            best_iters.append(int(best_iter))

            pred = model.predict(X[va_idx], num_iteration=best_iter)
            fold_rows.append(reg_metrics(y[va_idx], pred))

        fold_df = pd.DataFrame(fold_rows)
        rows.append({
            **cfg,
            "cfg_id": cfg_i,
            "folds": len(fold_rows),
            "mean_spearman": float(fold_df["Spearman"].mean()),
            "std_spearman": float(fold_df["Spearman"].std(ddof=0)),
            "mean_r2": float(fold_df["R2"].mean()),
            "mean_rmse": float(fold_df["RMSE"].mean()),
            "mean_mae": float(fold_df["MAE"].mean()),
            "robust_score": float(
                fold_df["Spearman"].mean()
                - robustness_penalty * fold_df["Spearman"].std(ddof=0)
            ),
            "median_best_iteration": int(np.median(best_iters)),
            "fold_metrics": fold_rows,
        })

    results = pd.DataFrame(rows).sort_values(
        ["robust_score", "mean_spearman", "mean_r2"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    best = results.iloc[0].to_dict()
    return results, best

In [7]:
# Core configuration
from Learn.features import _add_features_US500, _add_features_EURUSD

DS_NAME = "../data/US500_M1_520weeks.csv"
N_ROWS = 2_000_000
USE_CUSTOM_FEATURE_FUNCTION = True

TP_MULT = 2.5
SL_MULT = 2.5
TARGET_CLIP_MAX = max(TP_MULT, SL_MULT) * 1.5
TRAIN_FRACTION = 0.90
MAX_TUNING_ROWS = 350_000
WFO_FOLDS = 4
WFO_MIN_TRAIN_FRAC = 0.55
WFO_TEST_FRAC = 0.10
WFO_GAP_ROWS = 30
EARLY_STOPPING_ROUNDS = 200
ROBUSTNESS_PENALTY = 0.25
USE_TRADEABILITY_WEIGHTING = False

TARGET_COLS = ["long_quality", "short_quality", "signed_quality"]
PRIMARY_TARGET = "signed_quality"

regime_params = {
    "ma_period": 90,
    "slope_smoothness": 50,
    "regime_min_duration": 0,
    "atr_window": 60,
    "atr_lookback": 720,
    "atr_percentile": 0.0,
    "slope_threshold": 0,
}

outcome_params = {
    "atr_window": 60,
    "tp_mult": TP_MULT,
    "sl_mult": SL_MULT,
    "max_horizon": 30,
}

LGBM_BASE_PARAMS = {
    "objective": "regression",
    "boosting_type": "gbdt",
    "n_estimators": 5000,
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1,
    "max_depth": -1,
    "subsample_freq": 1,
}

LGBM_CANDIDATES = [
    dict(learning_rate=0.03, num_leaves=63, min_child_samples=60, subsample=0.80, colsample_bytree=0.80, reg_alpha=0.10, reg_lambda=0.50, min_split_gain=0.00),
    dict(learning_rate=0.02, num_leaves=127, min_child_samples=80, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.20, reg_lambda=1.00, min_split_gain=0.00),
    dict(learning_rate=0.01, num_leaves=255, min_child_samples=120, subsample=0.90, colsample_bytree=0.90, reg_alpha=0.50, reg_lambda=3.00, min_split_gain=0.00),
    dict(learning_rate=0.05, num_leaves=31, min_child_samples=100, subsample=0.75, colsample_bytree=0.75, reg_alpha=0.05, reg_lambda=5.00, min_split_gain=0.00),
    dict(learning_rate=0.025, num_leaves=63, min_child_samples=40, subsample=0.85, colsample_bytree=0.80, reg_alpha=0.10, reg_lambda=2.00, min_split_gain=0.01),
    dict(learning_rate=0.02, num_leaves=95, min_child_samples=50, subsample=0.80, colsample_bytree=0.75, reg_alpha=0.00, reg_lambda=1.00, min_split_gain=0.00),
]

model_source = _add_features_US500 if (USE_CUSTOM_FEATURE_FUNCTION and _add_features_US500 is not None) else add_feature_library
FEATURES = model_source

print("Device: CPU (tree model)")
print("Dataset:", DS_NAME)
print("Targets:", TARGET_COLS)
print("Feature function:", getattr(FEATURES, "__name__", str(FEATURES)))

Device: CPU (tree model)
Dataset: ../data/US500_M1_520weeks.csv
Targets: ['long_quality', 'short_quality', 'signed_quality']
Feature function: _add_features_US500


In [8]:
# Load data and compute target variables
from talib import ATR

df = load_ohlcv(DS_NAME, n_rows=N_ROWS)
print(f"Loaded {len(df):,} rows")
print(df["Time"].min(), "->", df["Time"].max())

df = FEATURES(df.copy(), include_mtf=True, regime_params=regime_params)
print('Added features')
df["atr"] = ATR(df["High"], df["Low"], df["Close"], timeperiod=14)
print('Added ATR')

outcomes = calculate_trade_outcomes_all_candles(df, **outcome_params)
print(f"Calculated trade outcomes for {len(outcomes):,} rows")

eps = 1e-8
long_q = np.log1p(outcomes["buy_MFE"] / (outcomes["buy_MAE"] + eps))
short_q = np.log1p(outcomes["sell_MFE"] / (outcomes["sell_MAE"] + eps))
print(f"Computed long and short quality metrics for {len(long_q):,} rows")

if TARGET_CLIP_MAX is not None:
    long_q = np.clip(long_q, 0.0, TARGET_CLIP_MAX)
    short_q = np.clip(short_q, 0.0, TARGET_CLIP_MAX)
print(f"Clipped long and short quality metrics to max {TARGET_CLIP_MAX}")

df["long_quality"] = long_q
df["short_quality"] = short_q
df["signed_quality"] = df["long_quality"] - df["short_quality"]
df["tradeability_score"] = 1.0

print("Target summary:")
display(describe_target_frame(df, TARGET_COLS))

print(
    f"Zero share | long={(df['long_quality'] <= 1e-12).mean():.3f} "
    f"short={(df['short_quality'] <= 1e-12).mean():.3f} "
    f"signed={(df['signed_quality'] == 0).mean():.3f}"
)

Loaded 2,000,000 rows
2020-10-02 18:53:00+00:00 -> 2026-06-19 16:59:00+00:00
Added features
Added ATR
Calculated trade outcomes for 2,000,000 rows
Computed long and short quality metrics for 2,000,000 rows
Clipped long and short quality metrics to max 3.75
Target summary:


,mean,std,p01,p50,p99
target,,,,,
long_quality,0.728114,0.828001,0.00,0.505310,3.75
short_quality,0.698909,0.812890,0.00,0.469092,3.75
signed_quality,0.029205,1.454657,-3.75,0.122147,3.75


Zero share | long=0.237 short=0.246 signed=0.005


In [9]:
# Preprocess features and create train / holdout splits
preprocess_ohlcv_args = {
    "target_col": TARGET_COLS + ["tradeability_score"],
    "outcomes_col": None,
    "shift": 0,
    "onehot_prefixes": ["OH_"],
    "price_prefixes": ["PR_"],
}

split_idx = int(len(df) * TRAIN_FRACTION)
df_train = df.iloc[:split_idx].copy().reset_index(drop=True)
df_holdout = df.iloc[split_idx:].copy().reset_index(drop=True)

X_train, y_train, scaler, features, _, proc_df_train = preprocess_ohlcv(
    df_train, **preprocess_ohlcv_args, scaler=None, return_df=True,
)
X_holdout, y_holdout, _, _, _, proc_df_holdout = preprocess_ohlcv(
    df_holdout, **preprocess_ohlcv_args, scaler=scaler, return_df=True,
)

target_arrays = {
    target: {
        "train": y_train[:, i],
        "holdout": y_holdout[:, i],
    }
    for i, target in enumerate(TARGET_COLS)
}

train_weight = None if not USE_TRADEABILITY_WEIGHTING else y_train[:, 3]

tune_rows = min(len(X_train), MAX_TUNING_ROWS)
X_tune = X_train[-tune_rows:]
tune_targets = {k: v["train"][-tune_rows:] for k, v in target_arrays.items()}
tune_weight = None if train_weight is None else train_weight[-tune_rows:]

tune_test_size = max(20_000, int(tune_rows * WFO_TEST_FRAC))
tune_min_train = max(100_000, int(tune_rows * WFO_MIN_TRAIN_FRAC))
tune_min_train = min(tune_min_train, tune_rows - tune_test_size - WFO_GAP_ROWS)
while tune_min_train + tune_test_size + WFO_GAP_ROWS > tune_rows and tune_test_size > 10_000:
    tune_test_size = int(tune_test_size * 0.8)
    tune_min_train = min(tune_min_train, tune_rows - tune_test_size - WFO_GAP_ROWS)

wf_folds = walk_forward_folds(
    n_samples=tune_rows,
    min_train_size=tune_min_train,
    test_size=tune_test_size,
    gap_size=max(WFO_GAP_ROWS, outcome_params["max_horizon"]),
    n_folds=WFO_FOLDS,
)

print(f"Train rows   : {len(X_train):,}")
print(f"Holdout rows : {len(X_holdout):,}")
print(f"Tuning rows  : {len(X_tune):,}")
print(f"Walk-forward folds: {len(wf_folds)}")
for i, (tr, va) in enumerate(wf_folds, start=1):
    print(f"  fold {i}: train={len(tr):,} val={len(va):,}")

Train rows   : 1,799,195
Holdout rows : 199,999
Tuning rows  : 350,000
Walk-forward folds: 4
  fold 1: train=192,500 val=35,000
  fold 2: train=227,530 val=35,000
  fold 3: train=262,560 val=35,000
  fold 4: train=297,590 val=35,000


In [10]:
# Tune hyper-parameters for each target
tuning_results = {}
best_config_by_target = {}
best_iteration_by_target = {}

for target_name in TARGET_COLS:
    print(f"\nTuning target: {target_name}")
    results_df, best_cfg = tune_lgbm_by_spearman(
        X=X_tune,
        y=tune_targets[target_name],
        folds=wf_folds,
        candidate_params=LGBM_CANDIDATES,
        base_params=LGBM_BASE_PARAMS,
        sample_weight=tune_weight,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        robustness_penalty=ROBUSTNESS_PENALTY,
    )
    tuning_results[target_name] = results_df
    best_config_by_target[target_name] = best_cfg
    best_iteration_by_target[target_name] = int(best_cfg["median_best_iteration"])

    print("Top candidates:")
    display(results_df[[
        "cfg_id", "mean_spearman", "std_spearman", "robust_score",
        "mean_r2", "mean_rmse", "median_best_iteration",
        "learning_rate", "num_leaves", "min_child_samples",
        "subsample", "colsample_bytree", "reg_alpha", "reg_lambda",
    ]].head(5).round(4))

tuning_summary = pd.DataFrame([
    {
        "target": target,
        "best_cfg_id": best_config_by_target[target]["cfg_id"],
        "best_mean_spearman": best_config_by_target[target]["mean_spearman"],
        "best_std_spearman": best_config_by_target[target]["std_spearman"],
        "best_robust_score": best_config_by_target[target]["robust_score"],
        "best_n_estimators": best_iteration_by_target[target],
    }
    for target in TARGET_COLS
]).set_index("target")

print("\nSelected tuning summary:")
display(tuning_summary.round(4))


Tuning target: long_quality
Top candidates:


,cfg_id,mean_spearman,std_spearman,robust_score,mean_r2,mean_rmse,median_best_iteration,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,4,0.3157,0.0072,0.3139,0.0929,0.8022,89,0.050,31,100,0.75,0.75,0.05,5.0
1,1,0.3140,0.0077,0.3120,0.0902,0.8034,75,0.030,63,60,0.80,0.80,0.10,0.5
2,5,0.3133,0.0079,0.3113,0.0851,0.8055,109,0.025,63,40,0.85,0.80,0.10,2.0
3,6,0.3128,0.0071,0.3111,0.0848,0.8057,91,0.020,95,50,0.80,0.75,0.00,1.0
4,2,0.3123,0.0071,0.3105,0.0856,0.8054,88,0.020,127,80,0.85,0.85,0.20,1.0



Tuning target: short_quality
Top candidates:


,cfg_id,mean_spearman,std_spearman,robust_score,mean_r2,mean_rmse,median_best_iteration,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,4,0.2900,0.0066,0.2883,0.0808,0.7860,65,0.050,31,100,0.75,0.75,0.05,5.0
1,1,0.2901,0.0075,0.2883,0.0777,0.7873,57,0.030,63,60,0.80,0.80,0.10,0.5
2,6,0.2898,0.0072,0.2880,0.0674,0.7918,79,0.020,95,50,0.80,0.75,0.00,1.0
3,2,0.2895,0.0062,0.2880,0.0666,0.7921,44,0.020,127,80,0.85,0.85,0.20,1.0
4,5,0.2898,0.0078,0.2878,0.0774,0.7874,78,0.025,63,40,0.85,0.80,0.10,2.0



Tuning target: signed_quality
Top candidates:


,cfg_id,mean_spearman,std_spearman,robust_score,mean_r2,mean_rmse,median_best_iteration,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,4,0.2568,0.0035,0.2560,0.0764,1.4154,52,0.050,31,100,0.75,0.75,0.05,5.0
1,5,0.2548,0.0028,0.2541,0.0536,1.4329,48,0.025,63,40,0.85,0.80,0.10,2.0
2,1,0.2541,0.0019,0.2537,0.0523,1.4333,36,0.030,63,60,0.80,0.80,0.10,0.5
3,6,0.2542,0.0035,0.2533,0.0579,1.4294,42,0.020,95,50,0.80,0.75,0.00,1.0
4,2,0.2528,0.0029,0.2521,0.0523,1.4333,63,0.020,127,80,0.85,0.85,0.20,1.0



Selected tuning summary:


,best_cfg_id,best_mean_spearman,best_std_spearman,best_robust_score,best_n_estimators
target,,,,,
long_quality,4,0.3157,0.0072,0.3139,89
short_quality,4,0.2900,0.0066,0.2883,65
signed_quality,4,0.2568,0.0035,0.2560,52


In [11]:
# Train final models on full training set and evaluate on holdout
final_models = {}
holdout_results = {}

for target_name in TARGET_COLS:
    best_cfg = best_config_by_target[target_name]
    final_params = {
        **LGBM_BASE_PARAMS,
        **{k: v for k, v in best_cfg.items() if k in LGBM_CANDIDATES[0]},
        "n_estimators": int(best_iteration_by_target[target_name]),
    }

    print(f"Training final model for {target_name} with {final_params['n_estimators']} trees")
    model = LGBMRegressor(**final_params)
    fit_kwargs = {"X": X_train, "y": target_arrays[target_name]["train"]}
    if train_weight is not None:
        fit_kwargs["sample_weight"] = train_weight
    model.fit(**fit_kwargs)

    pred_holdout = model.predict(X_holdout)
    final_models[target_name] = model
    holdout_results[target_name] = reg_metrics(
        target_arrays[target_name]["holdout"], pred_holdout
    )
    holdout_results[target_name]["best_n_estimators"] = int(final_params["n_estimators"])

signed_pred = final_models["signed_quality"].predict(X_holdout)
long_pred = final_models["long_quality"].predict(X_holdout)
short_pred = final_models["short_quality"].predict(X_holdout)
spread_pred = long_pred - short_pred

holdout_results["spread_proxy"] = reg_metrics(
    target_arrays["signed_quality"]["holdout"], spread_pred
)
holdout_results["spread_proxy"]["best_n_estimators"] = int(
    np.median([
        best_iteration_by_target["long_quality"],
        best_iteration_by_target["short_quality"],
    ])
)

holdout_df = pd.DataFrame(holdout_results).T
print("\nHoldout performance:")
display(holdout_df.round(4))

cv_best = pd.DataFrame([
    {
        "target": target,
        "cv_mean_spearman": tuning_summary.loc[target, "best_mean_spearman"],
        "cv_std_spearman": tuning_summary.loc[target, "best_std_spearman"],
        "cv_robust_score": tuning_summary.loc[target, "best_robust_score"],
        "holdout_spearman": holdout_results[target]["Spearman"],
        "holdout_r2": holdout_results[target]["R2"],
    }
    for target in TARGET_COLS
]).set_index("target")

print("\nCV vs holdout:")
display(cv_best.round(4))

Training final model for long_quality with 89 trees
Training final model for short_quality with 65 trees
Training final model for signed_quality with 52 trees

Holdout performance:


,MAE,RMSE,R2,Spearman,best_n_estimators
long_quality,0.5537,0.7219,0.0956,0.3107,89.0
short_quality,0.5482,0.7174,0.0902,0.3017,65.0
signed_quality,1.0398,1.2929,0.0856,0.2634,52.0
spread_proxy,1.0394,1.2915,0.0876,0.2642,77.0



CV vs holdout:


,cv_mean_spearman,cv_std_spearman,cv_robust_score,holdout_spearman,holdout_r2
target,,,,,
long_quality,0.3157,0.0072,0.3139,0.3107,0.0956
short_quality,0.2900,0.0066,0.2883,0.3017,0.0902
signed_quality,0.2568,0.0035,0.2560,0.2634,0.0856


In [12]:
# Trading-style proxy analysis on the signed target
y_true_signed = target_arrays["signed_quality"]["holdout"]
y_pred_signed = signed_pred
thresholds = np.quantile(np.abs(y_pred_signed), np.linspace(0.50, 0.95, 10))

sweep_rows = []
for thr in thresholds:
    pred_sign = np.zeros(len(y_pred_signed), dtype=int)
    pred_sign[y_pred_signed >= thr] = 1
    pred_sign[y_pred_signed <= -thr] = -1
    taken = pred_sign != 0

    if taken.sum() == 0:
        sweep_rows.append({
            "threshold": float(thr),
            "coverage": 0.0,
            "taken": 0,
            "long_count": 0,
            "short_count": 0,
            "sign_hit_rate": np.nan,
            "aligned_signed_mean": np.nan,
            "taken_spearman": np.nan,
        })
        continue

    sign_hit = np.mean(np.sign(y_true_signed[taken]) == pred_sign[taken])
    aligned_signed = np.where(
        pred_sign[taken] == 1, y_true_signed[taken], -y_true_signed[taken]
    )
    sweep_rows.append({
        "threshold": float(thr),
        "coverage": float(taken.mean()),
        "taken": int(taken.sum()),
        "long_count": int((pred_sign == 1).sum()),
        "short_count": int((pred_sign == -1).sum()),
        "sign_hit_rate": float(sign_hit),
        "aligned_signed_mean": float(np.mean(aligned_signed)),
        "taken_spearman": float(safe_spearman(y_true_signed[taken], y_pred_signed[taken])),
    })

threshold_df = pd.DataFrame(sweep_rows)
print("Signed-model threshold sweep:")
display(threshold_df.round(4))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=threshold_df["threshold"],
    y=threshold_df["aligned_signed_mean"],
    mode="lines+markers",
    name="Aligned signed mean",
))
fig.add_trace(go.Scatter(
    x=threshold_df["threshold"],
    y=threshold_df["sign_hit_rate"],
    mode="lines+markers",
    name="Sign hit rate",
))
fig.update_layout(
    title="Trading Proxy vs Confidence Threshold",
    xaxis_title="Absolute prediction threshold",
    yaxis_title="Score",
    height=420,
)
fig.show()

Signed-model threshold sweep:


,threshold,coverage,taken,long_count,short_count,sign_hit_rate,aligned_signed_mean,taken_spearman
0,0.2806,0.50,100004,55063,44941,0.6147,0.5138,0.3432
1,0.3082,0.45,90000,49506,40494,0.6214,0.5364,0.3546
2,0.3370,0.40,80000,44161,35839,0.6283,0.5589,0.3660
3,0.3642,0.35,70000,38994,31006,0.6363,0.5816,0.3786
4,0.3908,0.30,60000,34315,25685,0.6448,0.6034,0.3915
5,0.4181,0.25,50000,29398,20602,0.6563,0.6305,0.4060
6,0.4485,0.20,40000,23457,16543,0.6712,0.6659,0.4246
7,0.4843,0.15,30000,17817,12183,0.6882,0.7028,0.4458
8,0.5297,0.10,20000,11615,8385,0.7143,0.7489,0.4810
9,0.5961,0.05,10000,5675,4325,0.7661,0.8370,0.5417


In [13]:
# Export the model pack
today = pd.Timestamp.now().strftime("%Y%m%d")
ds_title = DS_NAME.split("/")[-1].split(".")[0]
model_name = "_".join([ds_title, "LightGBM", today, "spearman_prod"])

model_pack_dir = Path("../ModelWorkbench/ModelPacks")
model_pack_dir.mkdir(parents=True, exist_ok=True)
model_pack_path = model_pack_dir / f"{model_name}_model.pkl"

model_info = {
    "dataset_name": ds_title,
    "dataset_dir": DS_NAME,
    "date_trained": today,
    "model_type": "LightGBMRegressionPack",
    "task": "regression",
    "primary_target": PRIMARY_TARGET,
    "targets": TARGET_COLS,
    "cv_scheme": {
        "type": "expanding_walk_forward",
        "folds": len(wf_folds),
        "gap_rows": max(WFO_GAP_ROWS, outcome_params["max_horizon"]),
        "tuning_rows": int(len(X_tune)),
        "holdout_fraction": TRAIN_FRACTION,
        "robustness_penalty": ROBUSTNESS_PENALTY,
    },
}

pack = {
    "model": final_models[PRIMARY_TARGET],
    "primary_model": final_models[PRIMARY_TARGET],
    "aux_models": {k: v for k, v in final_models.items() if k != PRIMARY_TARGET},
    "model_class": final_models[PRIMARY_TARGET].__class__,
    "model_class_source": inspect.getsource(final_models[PRIMARY_TARGET].__class__),
    "model_params": {
        target: {
            **LGBM_BASE_PARAMS,
            **{k: v for k, v in best_config_by_target[target].items() if k in LGBM_CANDIDATES[0]},
            "n_estimators": int(best_iteration_by_target[target]),
        }
        for target in TARGET_COLS
    },
    "model_info": model_info,
    "features": features,
    "feature_count": X_train.shape[1],
    "feature_function": FEATURES,
    "feature_function_source": inspect.getsource(FEATURES),
    "preprocess_function": preprocess_ohlcv,
    "preprocess_function_source": inspect.getsource(preprocess_ohlcv),
    "preprocess_args": preprocess_ohlcv_args,
    "scaler": scaler,
    "label_function": calculate_trade_outcomes_all_candles,
    "label_function_source": inspect.getsource(calculate_trade_outcomes_all_candles),
    "label_params": None,
    "regime_params": regime_params,
    "outcome_params": outcome_params,
    "trading_hours": None,
    "input_shape": None,
    "data_split": {
        "train_rows": len(X_train),
        "holdout_rows": len(X_holdout),
        "train_fraction": TRAIN_FRACTION,
        "max_tuning_rows": MAX_TUNING_ROWS,
    },
    "validation": {
        "cv_summary": tuning_summary.to_dict(orient="index"),
        "cv_results": {
            target: tuning_results[target].head(10).to_dict(orient="records")
            for target in TARGET_COLS
        },
        "holdout": {
            target: {k: float(v) for k, v in holdout_results[target].items()}
            for target in holdout_results
        },
        "threshold_sweep": threshold_df.to_dict(orient="records"),
    },
}

with open(model_pack_path, "wb") as fh:
    pickle.dump(pack, fh)

print(f"Saved model pack to {model_pack_path}")

Saved model pack to ../ModelWorkbench/ModelPacks/US500_M1_520weeks_LightGBM_20260701_spearman_prod_model.pkl


In [14]:
# Model importance snapshot for the primary deployment model
importance_df = pd.DataFrame({
    "feature": features,
    "importance": final_models[PRIMARY_TARGET].feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

display(importance_df.head(30))

fig = go.Figure(go.Bar(
    x=importance_df.head(30)["importance"][::-1],
    y=importance_df.head(30)["feature"][::-1],
    orientation="h",
))
fig.update_layout(
    title=f"Top 30 Feature Importances - {PRIMARY_TARGET}",
    height=700,
    xaxis_title="Gain / split importance",
    yaxis_title="Feature",
)
fig.show()

,feature,importance
0,fl_close_loc,337
1,fl_hl_range_atr,251
2,fl_atr_ratio_14,190
3,MTF_30min_adx,100
4,MTF_30min_rsi,96
5,fl_hour_cos,68
6,MTF_30min_time_in_trend,67
7,MTF_15min_slope,64
8,MTF_30min_roc,63
9,fl_hour_sin,58
